**Imports & Configuration**
Imports libraries and defines paths, filenames, and NLLB language codes for translation direction.

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Configuration
CHECKPOINT_PATH = "nllb-specialization-5e4-do-0.3-5epochs/checkpoint-9380"
EXCEL_FILE = "dataset/inference-p1-p2-merged-deduplicated-80k.xlsx"
OUTPUT_FILE = "dataset/final-inferenced-parallel-corpus.xlsx"

SRC_LANG = "npi_Deva"  # Nepali Devanagari
TGT_LANG = "hin_Deva"  # Hindi Devanagari

**Load Model and Tokenizer from Checkpoint**
Loads the fine-tuned NLLB model and tokenizer from a local checkpoint and moves it to GPU if available.

In [ ]:
def load_model(checkpoint_path):
    print(f"Loading model from {checkpoint_path}...")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    print(f"Model loaded on {device}")

    return tokenizer, model, device

**Batch Translation Function**
Translates a list of sentences in batches using beam search and forced target language token (critical for NLLB).


In [ ]:
def translate_batch(texts, tokenizer, model, device, batch_size=8):
    translations = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        tokenizer.src_lang = SRC_LANG
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
                max_length=128,
                num_beams=5,
                early_stopping=True
            )

        batch_translations = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        translations.extend(batch_translations)

        print(f"Translated {min(i+batch_size, len(texts))}/{len(texts)} sentences")

    return translations

**Main Inference Pipeline**
Loads data from Excel, translates the relevant_sentences column, adds translations as new column translation_tmg, preserves all original columns, and saves output and shows sample results

In [ ]:
def main():
    tokenizer, model, device = load_model(CHECKPOINT_PATH)

    print(f"\nReading Excel file: {EXCEL_FILE}")
    df = pd.read_excel(EXCEL_FILE)

    if 'relevant_sentences' not in df.columns:
        raise ValueError("Column 'relevant_sentences' not found in Excel file")

    sentences = df['relevant_sentences'].fillna("").tolist()

    print(f"\nTranslating {len(sentences)} sentences from Nepali to Hindi...")
    translations = translate_batch(sentences, tokenizer, model, device)

    df['translation_tmg'] = translations

    print(f"\nSaving results to {OUTPUT_FILE}")
    df.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')

    print("Translation complete!")
    print(f"\nSample translations:")
    for i in range(min(3, len(df))):
        print(f"\nOriginal: {df.loc[i, 'relevant_sentences']}")
        print(f"Translation: {df.loc[i, 'translation_tmg']}")

**Entry Point**
Runs the full inference pipeline when script is executed directly.

In [ ]:
if __name__ == "__main__":
    main()